<a href="https://colab.research.google.com/github/viviantothewu-web/trashnet-waste-classification/blob/main/Vivian_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vivian Experiments — Seed 0

This notebook contains Vivian's model training experiments for the TrashNet waste-classification project.

The goal is to train and evaluate the shared CNN architectures using the team's fixed dataset split and shared training pipeline.

For Vivian's experiments:

- **Training seed:** 0
- **Image size:** 224 × 224
- **Batch size:** 32
- **Dataset:** TrashNet
- **Classes:** cardboard, glass, metal, paper, plastic, trash

The pretrained models trained in this notebook are:

- VGG16
- ResNet50
- MobileNetV2

All models use the same shared preprocessing, training, and transfer-learning code from the team's `src/` folder. This keeps the experiment consistent across team members and architectures.

The Custom CNN will be added later once the team finalizes the shared Custom CNN architecture.

The best model checkpoints will be selected using validation performance and saved for later evaluation on the locked TrashNet test set and the real-world waste-photo dataset.

In [3]:
# load team project from github so we can use the fixed data split and reusable pipeline modules.

from pathlib import Path

if not Path("trashnet-waste-classification").exists():
    !git clone https://github.com/viviantothewu-web/trashnet-waste-classification

print("Project repository loaded.")

Cloning into 'trashnet-waste-classification'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 111 (delta 54), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 10.91 MiB | 10.49 MiB/s, done.
Resolving deltas: 100% (54/54), done.
Project repository loaded.


In [4]:
# move into folder
import os

os.chdir("trashnet-waste-classification")

print("Current working directory:")
print(os.getcwd())

Current working directory:
/content/trashnet-waste-classification


In [5]:
# Clone our github repo

!git clone https://github.com/viviantothewu-web/trashnet-waste-classification

Cloning into 'trashnet-waste-classification'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 111 (delta 54), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 10.91 MiB | 10.73 MiB/s, done.
Resolving deltas: 100% (54/54), done.


In [6]:
#move to repo
import os

os.chdir("trashnet-waste-classification")

print("Current folder:", os.getcwd())

Current folder: /content/trashnet-waste-classification/trashnet-waste-classification


In [7]:
# Import shared project code

from src.data_pipeline import create_dataloaders
from src.transfer_learning import run_transfer_learning

print("imported")

imported


In [8]:
import random
import numpy as np
import torch

TRAINING_SEED = 0

random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAINING_SEED)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Training seed:", TRAINING_SEED)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Training seed: 0
Device: cuda
GPU: Tesla T4


In [9]:
# Download TrashNet and extract the resized dataset

!git clone https://github.com/garythung/trashnet.git

!unzip -q trashnet/data/dataset-resized.zip -d trashnet/data/

Cloning into 'trashnet'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 45 (delta 14), reused 8 (delta 8), pack-reused 24 (from 1)
Receiving objects: 100% (45/45), 40.64 MiB | 12.78 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [10]:
# Create TrashNet train, validation, and test loaders
#using shared data_pipeline.py instead of rebuilding the code here

IMG_SIZE = 224
BATCH_SIZE = 32

train_loader, val_loader, test_loader = create_dataloaders(
    split_csv_path="data_splits/trashnet_split.csv",
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    training_seed=TRAINING_SEED
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 56
Validation batches: 12
Test batches: 12


In [11]:
#check

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First few labels:", labels[:10])

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
First few labels: tensor([1, 4, 1, 3, 2, 3, 0, 2, 5, 3])


## VGG16 — Seed 0

This experiment trains VGG16 on the fixed TrashNet training split using assigned training seed of 0.

VGG16 starts with pretrained ImageNet weights and uses the team's shared two-stage transfer-learning procedure:

- **Stage 1:** Freeze the pretrained layers and train the new 6-class classifier for 5 epochs using a learning rate of 0.001.
- **Stage 2:** Unfreeze the model and fine-tune all layers for 5 epochs using a smaller learning rate of 0.0001.
- The best validation checkpoint is saved during each stage.

The resulting model will later be evaluated on the locked TrashNet test set and the real-world waste-photo dataset.

In [12]:
# Train VGG16 using seed 0

vgg_model, vgg_stage1_history, vgg_stage2_history = run_transfer_learning(
    model_name="vgg16",
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    training_seed=TRAINING_SEED
)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 160MB/s]


Epoch 1/5 | Train Loss: 1.1772 | Train Acc: 0.5430 | Val Loss: 1.0245 | Val Acc: 0.5989
  Best model saved (Val Acc: 0.5989)
Epoch 2/5 | Train Loss: 0.8790 | Train Acc: 0.6674 | Val Loss: 0.9079 | Val Acc: 0.6438
  Best model saved (Val Acc: 0.6438)
Epoch 3/5 | Train Loss: 0.8120 | Train Acc: 0.6963 | Val Loss: 0.8954 | Val Acc: 0.6702
  Best model saved (Val Acc: 0.6702)
Epoch 4/5 | Train Loss: 0.7803 | Train Acc: 0.6946 | Val Loss: 0.8605 | Val Acc: 0.6755
  Best model saved (Val Acc: 0.6755)
Epoch 5/5 | Train Loss: 0.7270 | Train Acc: 0.7245 | Val Loss: 0.7970 | Val Acc: 0.6992
  Best model saved (Val Acc: 0.6992)
Epoch 1/5 | Train Loss: 0.7120 | Train Acc: 0.7426 | Val Loss: 0.8489 | Val Acc: 0.6992
  Best model saved (Val Acc: 0.6992)
Epoch 2/5 | Train Loss: 0.4434 | Train Acc: 0.8501 | Val Loss: 0.5942 | Val Acc: 0.8100
  Best model saved (Val Acc: 0.8100)
Epoch 3/5 | Train Loss: 0.2996 | Train Acc: 0.8959 | Val Loss: 0.6056 | Val Acc: 0.7968
Epoch 4/5 | Train Loss: 0.2227 | Trai

so this basically:

* load pretrained VGG16
* replace the final classifier with 6 outputs
* Stage 1: train classifier for 5 epochs
* reload the best Stage 1 checkpoint
* unfreeze the model
* Stage 2: fine-tune for 5 epochs
* save the best Stage 2 checkpoint

In [13]:
# VGG16 training results
# validation only, no test yet

best_stage1_val_acc = max(vgg_stage1_history["val_accuracy"])
best_stage2_val_acc = max(vgg_stage2_history["val_accuracy"])

print("VGG16 Results - Seed 0")
print(f"Best Stage 1 validation accuracy: {best_stage1_val_acc:.4f}")
print(f"Best Stage 2 validation accuracy: {best_stage2_val_acc:.4f}")

VGG16 Results - Seed 0
Best Stage 1 validation accuracy: 0.6992
Best Stage 2 validation accuracy: 0.8364


## ResNet50 — Seed 0

This experiment trains ResNet50 on the same fixed TrashNet split using assigned training seed of 0.

ResNet50 uses pretrained ImageNet weights and the same shared two-stage transfer-learning procedure used for VGG16:

- **Stage 1:** Freeze the pretrained layers and train the new 6-class classifier for 5 epochs using a learning rate of 0.001.
- **Stage 2:** Unfreeze the model and fine-tune all layers for 5 epochs using a learning rate of 0.0001.
- The best validation checkpoint is saved during each stage.

Using the same training procedure keeps the comparison between VGG16 and ResNet50 fair.

In [14]:
#same process
resnet_model, resnet_stage1_history, resnet_stage2_history = run_transfer_learning(
    model_name="resnet50",
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    training_seed=TRAINING_SEED
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 172MB/s]


Epoch 1/5 | Train Loss: 1.2441 | Train Acc: 0.6069 | Val Loss: 0.9350 | Val Acc: 0.7599
  Best model saved (Val Acc: 0.7599)
Epoch 2/5 | Train Loss: 0.7976 | Train Acc: 0.7726 | Val Loss: 0.7265 | Val Acc: 0.7889
  Best model saved (Val Acc: 0.7889)
Epoch 3/5 | Train Loss: 0.6428 | Train Acc: 0.8252 | Val Loss: 0.6573 | Val Acc: 0.7889
Epoch 4/5 | Train Loss: 0.5573 | Train Acc: 0.8490 | Val Loss: 0.5793 | Val Acc: 0.8285
  Best model saved (Val Acc: 0.8285)
Epoch 5/5 | Train Loss: 0.4964 | Train Acc: 0.8648 | Val Loss: 0.5381 | Val Acc: 0.8285
Epoch 1/5 | Train Loss: 0.3423 | Train Acc: 0.8852 | Val Loss: 0.3348 | Val Acc: 0.8971
  Best model saved (Val Acc: 0.8971)
Epoch 2/5 | Train Loss: 0.1292 | Train Acc: 0.9627 | Val Loss: 0.3120 | Val Acc: 0.9050
  Best model saved (Val Acc: 0.9050)
Epoch 3/5 | Train Loss: 0.0607 | Train Acc: 0.9876 | Val Loss: 0.3005 | Val Acc: 0.9024
Epoch 4/5 | Train Loss: 0.0406 | Train Acc: 0.9910 | Val Loss: 0.2290 | Val Acc: 0.9182
  Best model saved (Val

In [15]:
# ResNet50 training results

best_stage1_val_acc = max(resnet_stage1_history["val_accuracy"])
best_stage2_val_acc = max(resnet_stage2_history["val_accuracy"])

print("ResNet50 Results - Seed 0")
print(f"Best Stage 1 validation accuracy: {best_stage1_val_acc:.4f}")
print(f"Best Stage 2 validation accuracy: {best_stage2_val_acc:.4f}")

ResNet50 Results - Seed 0
Best Stage 1 validation accuracy: 0.8285
Best Stage 2 validation accuracy: 0.9314


## MobileNetV2 — Seed 0

This experiment trains MobileNetV2 on the same fixed TrashNet split using assigned training seed of 0.

MobileNetV2 uses pretrained ImageNet weights and the same shared two-stage transfer-learning procedure used for VGG16 and ResNet50:

- **Stage 1:** Freeze the pretrained layers and train the new 6-class classifier for 5 epochs using a learning rate of 0.001.
- **Stage 2:** Unfreeze the model and fine-tune all layers for 5 epochs using a learning rate of 0.0001.
- The best validation checkpoint is saved during each stage.

Using the same training procedure keeps the comparison across all three pretrained architectures consistent.

In [16]:
#same process
mobilenet_model, mobilenet_stage1_history, mobilenet_stage2_history = run_transfer_learning(
    model_name="mobilenetv2",
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    training_seed=TRAINING_SEED
)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 187MB/s]


Epoch 1/5 | Train Loss: 1.3218 | Train Acc: 0.5436 | Val Loss: 1.0320 | Val Acc: 0.6728
  Best model saved (Val Acc: 0.6728)
Epoch 2/5 | Train Loss: 0.8828 | Train Acc: 0.7330 | Val Loss: 0.8117 | Val Acc: 0.7441
  Best model saved (Val Acc: 0.7441)
Epoch 3/5 | Train Loss: 0.7450 | Train Acc: 0.7726 | Val Loss: 0.7082 | Val Acc: 0.7678
  Best model saved (Val Acc: 0.7678)
Epoch 4/5 | Train Loss: 0.6644 | Train Acc: 0.7964 | Val Loss: 0.6446 | Val Acc: 0.7731
  Best model saved (Val Acc: 0.7731)
Epoch 5/5 | Train Loss: 0.6011 | Train Acc: 0.8105 | Val Loss: 0.6165 | Val Acc: 0.7836
  Best model saved (Val Acc: 0.7836)
Epoch 1/5 | Train Loss: 0.4859 | Train Acc: 0.8518 | Val Loss: 0.4523 | Val Acc: 0.8311
  Best model saved (Val Acc: 0.8311)
Epoch 2/5 | Train Loss: 0.3040 | Train Acc: 0.9095 | Val Loss: 0.3611 | Val Acc: 0.8786
  Best model saved (Val Acc: 0.8786)
Epoch 3/5 | Train Loss: 0.2055 | Train Acc: 0.9389 | Val Loss: 0.3277 | Val Acc: 0.8892
  Best model saved (Val Acc: 0.8892)


In [17]:
#MobileNetV2 training results
best_stage1_val_acc = max(mobilenet_stage1_history["val_accuracy"])
best_stage2_val_acc = max(mobilenet_stage2_history["val_accuracy"])

print("MobileNetV2 Results - Seed 0")
print(f"Best Stage 1 validation accuracy: {best_stage1_val_acc:.4f}")
print(f"Best Stage 2 validation accuracy: {best_stage2_val_acc:.4f}")

MobileNetV2 Results - Seed 0
Best Stage 1 validation accuracy: 0.7836
Best Stage 2 validation accuracy: 0.9077


- From here:

VGG16, ResNet50, and MobileNetV2 all trained

- Next:
1. Save/copy stage 1 and 2 checkpoints
5. Add Custom CNN later

In [18]:
from google.colab import files

checkpoint_files = [
    "saved_models/vgg16_seed0_stage1.pth",
    "saved_models/vgg16_seed0_stage2.pth",
    "saved_models/resnet50_seed0_stage1.pth",
    "saved_models/resnet50_seed0_stage2.pth",
    "saved_models/mobilenetv2_seed0_stage1.pth",
    "saved_models/mobilenetv2_seed0_stage2.pth"
]

for checkpoint in checkpoint_files:
    files.download(checkpoint)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (2 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Vivian - Seed 0 Summary

### What has been completed
completed training for the three pretrained CNN architectures using **training seed 0**:

- VGG16
- ResNet50
- MobileNetV2

All models used the team's shared:

- fixed TrashNet train/validation/test split
- image size: **224 × 224**
- batch size: **32**
- ImageNet normalization
- PyTorch training pipeline
- two-stage transfer-learning procedure

### Transfer-Learning Procedure

Each pretrained model was trained using:

**Stage 1**
- Load pretrained ImageNet weights.
- Replace the original classifier with a new **6-class TrashNet classifier**.
- Freeze the pretrained layers.
- Train only the new classifier for **5 epochs**.
- Learning rate: **0.001**.
- Save the checkpoint with the best validation accuracy.

**Stage 2**
- Reload the best Stage 1 checkpoint.
- Unfreeze the entire model.
- Fine-tune all layers for **5 epochs**.
- Learning rate: **0.0001**.
- Save the checkpoint with the best validation accuracy.

The **Stage 2 checkpoint is the final fine-tuned model** that should be used for the main TrashNet and real-world evaluations. Stage 1 checkpoints are preserved for reference.

### Saved Checkpoints

The checkpoints are stored in the shared Google Drive project folder under the folder "Seed_0"

Files:

- `vgg16_seed0_stage1.pth`
- `vgg16_seed0_stage2.pth`
- `resnet50_seed0_stage1.pth`
- `resnet50_seed0_stage2.pth`
- `mobilenetv2_seed0_stage1.pth`
- `mobilenetv2_seed0_stage2.pth`

For final evaluation, use:

- `vgg16_seed0_stage2.pth`
- `resnet50_seed0_stage2.pth`
- `mobilenetv2_seed0_stage2.pth`

---

## How to Load a Saved Checkpoint in case you guys need this

A `.pth` checkpoint contains the model's learned weights. To evaluate it, first recreate the same architecture and then load the saved weights.

Example for Vivian's ResNet50:

```python
import torch
from src.transfer_learning import create_pretrained_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate the same 6-class ResNet50 architecture
model = create_pretrained_model("resnet50")

# Path to Vivian's saved Stage 2 checkpoint
checkpoint_path = "PATH_TO/resnet50_seed0_stage2.pth"

# Load the trained weights
model.load_state_dict(
    torch.load(checkpoint_path, map_location=device)
)

# Move model to the available device
model = model.to(device)

# Put model in evaluation mode
model.eval()

print("Seed 0 checkpoint loaded successfully.")
```

Change the architecture name and checkpoint path for the other models:

```python
create_pretrained_model("vgg16")
create_pretrained_model("resnet50")
create_pretrained_model("mobilenetv2")
```

### Important

Loading a checkpoint **does not retrain the model**. It restores the weights I already learned during training.

During evaluation:

- use `model.eval()`
- do not call `loss.backward()`
- do not call `optimizer.step()`
- use the locked TrashNet **test set**
- use the same shared preprocessing pipeline
- do not change the saved model weights

---

## Next Team Step

The next step is to use one shared evaluation procedure for all team checkpoints.

The evaluation should eventually calculate:

- accuracy
- precision
- recall
- F1-score
- confusion matrix
- per-class performance
- inference time
- model size

The same evaluation procedure should be used for Vivian's seed 0, Joshua's seed 1, and Devan's seed 2 models so the results are directly comparable.

The locked TrashNet test set should be used for **Phase 1 benchmark evaluation**. The final trained models will later also be evaluated on the team's frozen real-world phone-photo dataset to measure the difference between benchmark and real-world generalization.